In [13]:
# 📦 Imports
import pandas as pd
import numpy as np
import random
import sys
import os

# 🔧 Set up path for src
sys.path.append(os.path.abspath('../src'))

# 🧠 Load Recommenders
from cf_model import CFRecommender
from content_model import ContentRecommender
from embedding_model import EmbeddingRecommender
from hybrid_model import HybridRecommender
from reranker import Reranker

# 📁 Load Processed Data
df = pd.read_pickle('../data/processed_playlists.pkl')
df_tracks = pd.read_pickle('../data/processed_tracks.pkl')

# --- Initialize Recommenders ---
cf = CFRecommender()
cf.prepare_matrix(df)
cf.train()

content = ContentRecommender()
content.prepare(df_tracks)

embedding = EmbeddingRecommender(vector_size=64, window=5, min_count=1)
playlists = embedding.prepare_training_data(df)
embedding.train(playlists)

hybrid = HybridRecommender(cf_model=cf, content_model=content, embedding_model=embedding, weights=(0.4, 0.3, 0.3))

# --- Choose a random playlist to recommend for ---
example_pid = df['playlist_id'].sample(1).iloc[0]
playlist_tracks = df[df['playlist_id'] == example_pid]['track_uri'].tolist()

print("🎧 Example Playlist ID:", example_pid)
print("📀 Original Playlist Tracks:")
display(df_tracks[df_tracks['track_uri'].isin(playlist_tracks)][['track_name', 'artist_name']])

# --- Hybrid recommendations with scores ---
recommendations_with_scores = hybrid.recommend_tracks(example_pid, playlist_tracks, top_n=15)

recommended_uris = [track for track, _ in recommendations_with_scores]
recommended_scores = [score for _, score in recommendations_with_scores]

print("\n🎯 Hybrid Recommendations (Before Reranking):")
display(df_tracks[df_tracks['track_uri'].isin(recommended_uris)][['track_name', 'artist_name']])

# --- Prepare popularity info ---
track_popularity = df['track_uri'].value_counts()

# --- Automatically identify underrepresented artists ---
# Define underrepresented as artists with median popularity below a threshold (e.g., bottom 10%)
artist_popularity = df_tracks[['artist_name', 'track_uri']].copy()
artist_popularity['popularity'] = artist_popularity['track_uri'].map(track_popularity).fillna(0)

# Group by artist and get median popularity of their tracks
artist_median_pop = artist_popularity.groupby('artist_name')['popularity'].median()

# Filter artists with positive median popularity only for threshold calculation
positive_medians = artist_median_pop[artist_median_pop > 0]

if positive_medians.empty:
    # fallback if no positives found
    pop_threshold = 0.01  
else:
    pop_threshold = positive_medians.quantile(0.10)

# Select underrepresented artists with median popularity <= threshold
underrepresented_artists = set(artist_median_pop[artist_median_pop <= pop_threshold].index)

print(f"Identified {len(underrepresented_artists)} underrepresented artists based on popularity threshold {pop_threshold:.4f}")


# --- Build user fairness profile ---
# Example: ratio of less popular tracks (<30th percentile popularity) in user's current playlist
user_track_pop = [track_popularity.get(uri, 0) for uri in playlist_tracks]
if user_track_pop:
    threshold_pop = np.percentile(track_popularity, 30)
    low_pop_count = sum(p < threshold_pop for p in user_track_pop)
    low_pop_ratio = low_pop_count / len(user_track_pop)
else:
    low_pop_ratio = 0.3  # fallback default

user_profile = {
    'low_popularity_ratio': low_pop_ratio
}
print(f"User fairness profile: prefers ~{low_pop_ratio:.2f} fraction of low popularity tracks")

# --- Build artist map for reranker ---
track_artist_map = dict(zip(df_tracks['track_uri'], df_tracks['artist_name']))

# --- Initialize fairness-aware reranker ---
reranker = Reranker(
    track_popularity=track_popularity,
    popularity_weight=0.7,
    diversity_weight=0.3,
    underrep_weight=0.5,
    user_fairness_weight=0.3,
    underrepresented_artists=underrepresented_artists,
    user_profile=user_profile
)
reranker.set_artist_map(track_artist_map)

# --- Rerank the recommendations ---
reranked_uris = reranker.rerank(recommended_uris, recommended_scores)

# --- Display reranked recommendations ---
print("\n🔥 Top 10 Reranked Recommendations:")
missing_uris = []

for uri in reranked_uris[:10]:
    track_info = df_tracks[df_tracks['track_uri'] == uri]
    
    if track_info.empty:
        # Try to find metadata from playlist data as fallback
        fallback_info = df[df['track_uri'] == uri][['track_uri', 'track_name', 'artist_name']].drop_duplicates()
        
        if not fallback_info.empty:
            name = fallback_info['track_name'].values[0]
            artist = fallback_info['artist_name'].values[0]
            print(f"✅ {name} by {artist} (from playlist data fallback)")
            
            # Optionally append this fallback metadata to df_tracks for future
            df_tracks = pd.concat([df_tracks, fallback_info], ignore_index=True)
            df_tracks.drop_duplicates(subset=['track_uri'], inplace=True)
            
        else:
            print(f"⚠️ Track URI '{uri}' not found in metadata or playlist data.")
            missing_uris.append(uri)
    else:
        name = track_info['track_name'].values[0]
        artist = track_info['artist_name'].values[0]
        print(f"✅ {name} by {artist}")

if missing_uris:
    print(f"\n❗ {len(missing_uris)} URIs still missing after fallback.")
else:
    print("\n✅ All reranked URIs found in metadata or playlist data fallback.")



🔄 Preparing user-item matrix...
Training collaborative filtering model...


  0%|          | 0/20 [00:00<?, ?it/s]

🎧 Example Playlist ID: 109467
📀 Original Playlist Tracks:


,track_name,artist_name



🎯 Hybrid Recommendations (Before Reranking):


,track_name,artist_name
4024,Novacane,Frank Ocean
17270,Biking,Frank Ocean
76098,You & I (feat. Tyler Sjöström & Bertrand Lacoste),Sterkøl
177989,B.S.D. (feat. Jasper Dolphin & Taco),"Tyler, The Creator"
359894,Frank Ocean,Mir Fontane
665836,Ocean,Tyler Pag
1858816,You & I (feat. Tyler Sjöström & Bertrand Lacos...,Sterkøl


Identified 107167 underrepresented artists based on popularity threshold 0.0100
User fairness profile: prefers ~0.00 fraction of low popularity tracks

🔥 Top 10 Reranked Recommendations:
✅ Lose Control (feat. Ciara & Fat Man Scoop) by Missy Elliott (from playlist data fallback)
✅ Redbone by Childish Gambino (from playlist data fallback)
✅ B.S.D. (feat. Jasper Dolphin & Taco) by Tyler, The Creator
✅ You & I (feat. Tyler Sjöström & Bertrand Lacoste) by Sterkøl
✅ I Thought by KYLE (from playlist data fallback)
✅ You & I (feat. Tyler Sjöström & Bertrand Lacoste) - VINIL Remix by Sterkøl
✅ Ocean by Tyler Pag
✅ Out of My Hands by Quentin Miller (from playlist data fallback)
✅ Frank Ocean by Mir Fontane
✅ Songs for Women by Frank Ocean (from playlist data fallback)

✅ All reranked URIs found in metadata or playlist data fallback.


In [14]:
import pickle

# Create a dictionary encapsulating everything needed
hybrid_config = {
    'cf_model': cf,
    'content_model': content,
    'embedding_model': embedding,
    'hybrid_model': hybrid,
    'reranker': reranker,
    'underrepresented_artists': underrepresented_artists,
    'user_profile': user_profile,
    'track_artist_map': track_artist_map
}

# Save to a pickle file
with open('../models/hybrid_recommender.pkl', 'wb') as f:
    pickle.dump(hybrid_config, f)

print("✅ Hybrid recommender and reranker configuration saved successfully.")


✅ Hybrid recommender and reranker configuration saved successfully.


In [15]:
import pickle

# Load the saved configuration
with open('../models/hybrid_recommender.pkl', 'rb') as f:
    config = pickle.load(f)

# Unpack models and objects
cf = config['cf_model']
content = config['content_model']
embedding = config['embedding_model']
hybrid = config['hybrid_model']
reranker = config['reranker']
underrepresented_artists = config['underrepresented_artists']
user_profile = config['user_profile']
track_artist_map = config['track_artist_map']

print("📦 Hybrid and reranker models loaded successfully.")


📦 Hybrid and reranker models loaded successfully.


In [16]:
from sklearn.metrics import precision_score, recall_score

def evaluate_model(hybrid, reranker, df, k=10, n_playlists=100):
    precisions = []
    recalls = []
    
    sampled_pids = df['playlist_id'].drop_duplicates().sample(n=min(n_playlists, df['playlist_id'].nunique()), random_state=42)
    
    for pid in sampled_pids:
        full_tracks = df[df['playlist_id'] == pid]['track_uri'].tolist()
        
        if len(full_tracks) < 5:
            continue  # Skip too short playlists
        
        # Split into seed and holdout
        seed_size = max(1, int(len(full_tracks) * 0.5))
        seed_tracks = full_tracks[:seed_size]
        holdout_tracks = full_tracks[seed_size:]
        
        # Get hybrid recommendations
        recs_with_scores = hybrid.recommend_tracks(pid, seed_tracks, top_n=100)
        rec_uris = [uri for uri, _ in recs_with_scores]
        # Rerank
        final_uris = reranker.rerank(rec_uris, [score for _, score in recs_with_scores])
        
        # Top-k
        top_k = final_uris[:k]
        
        # Compute Precision@K and Recall@K
        hits = [1 if uri in holdout_tracks else 0 for uri in top_k]
        
        if not holdout_tracks:
            continue
        
        precision = sum(hits) / k
        recall = sum(hits) / len(holdout_tracks)
        
        precisions.append(precision)
        recalls.append(recall)

    avg_precision = np.mean(precisions)
    avg_recall = np.mean(recalls)
    
    print(f"📈 Evaluation Results over {len(precisions)} playlists:")
    print(f"🔹 Precision@{k}: {avg_precision:.4f}")
    print(f"🔹 Recall@{k}:    {avg_recall:.4f}")

# Run evaluation
evaluate_model(hybrid, reranker, df, k=10, n_playlists=100)


📈 Evaluation Results over 100 playlists:
🔹 Precision@10: 0.0110
🔹 Recall@10:    0.0085


In [19]:
# --- Enhanced Evaluation with More Recommendations and Metrics ---

from sklearn.metrics import ndcg_score

def enhanced_evaluate(hybrid, df, df_tracks, reranker, top_n=30, n_eval=100):
    precisions, recalls, ndcgs = [], [], []

    # Sample n_eval playlists
    sample_pids = df['playlist_id'].drop_duplicates().sample(n_eval, random_state=42)

    for pid in sample_pids:
        playlist_tracks = df[df['playlist_id'] == pid]['track_uri'].tolist()
        if len(playlist_tracks) < 5:
            continue  # skip too small playlists
        
        # Split: last 2 tracks as test, rest as train
        train_tracks = playlist_tracks[:-2]
        test_tracks = set(playlist_tracks[-2:])

        # Get hybrid recommendations on train tracks
        recs_with_scores = hybrid.recommend_tracks(pid, train_tracks, top_n=top_n)
        rec_uris = [track for track, _ in recs_with_scores]
        rec_scores = [score for _, score in recs_with_scores]

        # Rerank recommendations
        reranked_uris = reranker.rerank(rec_uris, rec_scores)

        # Compute precision@10 and recall@10
        top_k = reranked_uris[:10]
        hits = [1 if uri in test_tracks else 0 for uri in top_k]

        precision = sum(hits) / 10
        recall = sum(hits) / len(test_tracks)

        # For ndcg: create relevance vector for all recs (top_n), 1 if in test else 0
        relevance = [1 if uri in test_tracks else 0 for uri in reranked_uris[:top_n]]
        ndcg = ndcg_score([relevance], [np.arange(top_n, 0, -1)])  # relevance vs ideal rank

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    print(f"📈 Enhanced Evaluation over {len(precisions)} playlists:")
    print(f"🔹 Avg Precision@10: {np.mean(precisions):.4f}")
    print(f"🔹 Avg Recall@10:    {np.mean(recalls):.4f}")
    print(f"🔹 Avg NDCG@10:      {np.mean(ndcgs):.4f}")

# Usage:
# Assuming your existing hybrid and reranker objects are defined
enhanced_evaluate(hybrid, df, df_tracks, reranker, top_n=30, n_eval=100)


📈 Enhanced Evaluation over 100 playlists:
🔹 Avg Precision@10: 0.0010
🔹 Avg Recall@10:    0.0050
🔹 Avg NDCG@10:      0.0088
